# Constants

In [1]:
LAHMAN_DIR = "../assets/lahman"


# Load Data Sets
Load Lahman datasets as DataFrames so they can be displayed and referenced.
Descriptions of the the columns can be found [here](../assets/lahman/readme2014.txt).

In [2]:
import pandas as pd
import os

from typing import Dict

lahman_dfs: Dict[str, pd.DataFrame] = {}

# Loop through all of the CSVs and create DFs
for csv in os.listdir(LAHMAN_DIR):
    name, ext = csv.split(".")

    if ext != "csv":
        continue

    df = pd.read_csv(os.path.join(LAHMAN_DIR, csv))

    lahman_dfs[name] = df


# Get Hall of Famers

Next, we will get the data surrounding the hall of famers.
We are only concerned with players, and we only need the last time that they appeared on the ballot.

In [3]:
# Display the Hall of Fame df
lahman_dfs["HallOfFame"].tail()


,playerID,yearID,votedBy,ballots,needed,votes,inducted,category,needed_note
4186,lidgebr01,2018,BBWAA,422.0,317.0,0.0,N,Player,NaN
4187,millwke01,2018,BBWAA,422.0,317.0,0.0,N,Player,NaN
4188,zambrca01,2018,BBWAA,422.0,317.0,0.0,N,Player,NaN
4189,morrija02,2018,Veterans,NaN,NaN,NaN,Y,Player,NaN
4190,trammal01,2018,Veterans,NaN,NaN,NaN,Y,Player,NaN


In [4]:
# A list of hall of fame players that were inducted into the HOF
lahman_dfs["HallOfFame"].loc[
    (lahman_dfs["HallOfFame"]["inducted"] == "Y")
    & (lahman_dfs["HallOfFame"]["category"] == "Player")
    & (lahman_dfs["HallOfFame"]["votedBy"] == "BBWAA")
]


,playerID,yearID,votedBy,ballots,needed,votes,inducted,category,needed_note
0,cobbty01,1936,BBWAA,226.0,170.0,222.0,Y,Player,NaN
1,ruthba01,1936,BBWAA,226.0,170.0,215.0,Y,Player,NaN
2,wagneho01,1936,BBWAA,226.0,170.0,215.0,Y,Player,NaN
3,mathech01,1936,BBWAA,226.0,170.0,205.0,Y,Player,NaN
4,johnswa01,1936,BBWAA,226.0,170.0,189.0,Y,Player,NaN
...,...,...,...,...,...,...,...,...,...
4122,rodriiv01,2017,BBWAA,442.0,332.0,336.0,Y,Player,NaN
4156,jonesch06,2018,BBWAA,422.0,317.0,410.0,Y,Player,NaN
4157,guerrvl01,2018,BBWAA,422.0,317.0,392.0,Y,Player,NaN
4158,thomeji01,2018,BBWAA,422.0,317.0,379.0,Y,Player,NaN


Get a DataFrame containing the last time a player has appeared on the ballot and the result

In [5]:
hof_df = lahman_dfs["HallOfFame"].drop(labels=["needed_note"], axis=1)
hof_df = hof_df.sort_values(by=["yearID"], ascending=True)
hof_df = hof_df.drop_duplicates(subset=["playerID"], keep="last")
hof_df = hof_df.loc[hof_df["category"] == "Player"]
hof_df


,playerID,yearID,votedBy,ballots,needed,votes,inducted,category
0,cobbty01,1936,BBWAA,226.0,170.0,222.0,Y,Player
79,doyleja01,1936,Veterans,78.0,59.0,1.0,N,Player
74,bondto01,1936,Veterans,78.0,59.0,1.0,N,Player
72,battijo01,1936,Veterans,78.0,59.0,1.0,N,Player
71,allisdo01,1936,Veterans,78.0,59.0,1.0,N,Player
...,...,...,...,...,...,...,...,...
4168,ramirma02,2018,BBWAA,422.0,317.0,93.0,N,Player
4169,kentje01,2018,BBWAA,422.0,317.0,61.0,N,Player
4170,sheffga01,2018,BBWAA,422.0,317.0,47.0,N,Player
4172,rolensc01,2018,BBWAA,422.0,317.0,43.0,N,Player


# Batting
We will now get the batting statistics for all the players.
We need to aggregate the stats for each player over their career, since HOF decisions are made by observing the entire career of a player.
We will do some feature engineering as well adding stats such as batting average, OBP, SLG, and OPS.
We will also drop other metrics such as GIPD and SH.

In [6]:
lahman_dfs["Batting"].columns


Index(['playerID', 'yearID', 'stint', 'teamID', 'lgID', 'G', 'AB', 'R', 'H',
       '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH',
       'SF', 'GIDP'],
      dtype='object')

In [7]:
batters_df = (
    lahman_dfs["Batting"][
        [
            "playerID",
            "G",
            "AB",
            "R",
            "H",
            "2B",
            "3B",
            "HR",
            "RBI",
            "SB",
            "CS",
            "BB",
            "SO",
            "IBB",
            "HBP",
            "SH",
            "SF",
            "GIDP",
        ]
    ]
    .groupby(["playerID"])
    .sum()
    .reset_index()
)

batters_df


,playerID,G,AB,R,H,2B,3B,HR,RBI,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP
0,aardsda01,331,4,0,0,0,0,0,0.0,0.0,0.0,0,2.0,0.0,0.0,1.0,0.0,0.0
1,aaronha01,3298,12364,2174,3771,624,98,755,2297.0,240.0,73.0,1402,1383.0,293.0,32.0,21.0,121.0,328.0
2,aaronto01,437,944,102,216,42,6,13,94.0,9.0,8.0,86,145.0,3.0,0.0,9.0,6.0,36.0
3,aasedo01,448,5,0,0,0,0,0,0.0,0.0,0.0,0,3.0,0.0,0.0,0.0,0.0,0.0
4,abadan01,15,21,1,2,0,0,0,0.0,0.0,1.0,4,5.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20161,zupofr01,16,18,3,3,1,0,0,0.0,0.0,0.0,2,6.0,0.0,0.0,0.0,0.0,0.0
20162,zuvelpa01,209,491,41,109,17,2,2,20.0,2.0,0.0,34,50.0,1.0,2.0,18.0,0.0,8.0
20163,zuverge01,266,142,5,21,2,1,0,7.0,0.0,1.0,9,39.0,0.0,0.0,16.0,0.0,3.0
20164,zwilldu01,366,1280,167,364,76,15,30,202.0,46.0,0.0,128,155.0,0.0,4.0,31.0,0.0,0.0


In [8]:
# Add OBP = (H + BB + HBP) / (AB + BB + HBP + SF)
batters_df["OBP"] = (batters_df["H"] + batters_df["BB"] + batters_df["HBP"]) / (
    batters_df["AB"] + batters_df["BB"] + batters_df["HBP"] + batters_df["SF"]
)

# Add SLG = (1B + 2*2B + 3*3B + 4*HR) / AB
batters_df["SLG"] = (
    batters_df["H"]
    - (batters_df["2B"] + batters_df["3B"] + batters_df["HR"])
    + 2 * batters_df["2B"]
    + 3 * batters_df["3B"]
    + 4 * batters_df["HR"]
) / batters_df["AB"]

# Add OPS = OBP + SLG
batters_df["OPS"] = batters_df["SLG"] + batters_df["OBP"]

# Add BA = H / AB
batters_df["BA"] = batters_df["H"] / batters_df["AB"]

# Drop GIPD and SH
batters_df = batters_df.drop(labels=["GIDP", "SH"], axis=1)

batters_df


,playerID,G,AB,R,H,2B,3B,HR,RBI,SB,CS,BB,SO,IBB,HBP,SF,OBP,SLG,OPS,BA
0,aardsda01,331,4,0,0,0,0,0,0.0,0.0,0.0,0,2.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
1,aaronha01,3298,12364,2174,3771,624,98,755,2297.0,240.0,73.0,1402,1383.0,293.0,32.0,121.0,0.373949,0.554513,0.928462,0.304998
2,aaronto01,437,944,102,216,42,6,13,94.0,9.0,8.0,86,145.0,3.0,0.0,6.0,0.291506,0.327331,0.618836,0.228814
3,aasedo01,448,5,0,0,0,0,0,0.0,0.0,0.0,0,3.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
4,abadan01,15,21,1,2,0,0,0,0.0,0.0,1.0,4,5.0,0.0,0.0,0.0,0.240000,0.095238,0.335238,0.095238
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20161,zupofr01,16,18,3,3,1,0,0,0.0,0.0,0.0,2,6.0,0.0,0.0,0.0,0.250000,0.222222,0.472222,0.166667
20162,zuvelpa01,209,491,41,109,17,2,2,20.0,2.0,0.0,34,50.0,1.0,2.0,0.0,0.275142,0.276986,0.552128,0.221996
20163,zuverge01,266,142,5,21,2,1,0,7.0,0.0,1.0,9,39.0,0.0,0.0,0.0,0.198675,0.176056,0.374732,0.147887
20164,zwilldu01,366,1280,167,364,76,15,30,202.0,46.0,0.0,128,155.0,0.0,4.0,0.0,0.351275,0.437500,0.788775,0.284375


# Fielding

I wanted to use fielding statistics to make sure defensive metrics are also included by the machine learning model.
While fielding is very important in the game of baseball, it seems to be overshadowed by offensive play.
This is not to say that fielding cannot lead to the selection of players in to the HOF.
A great example of a hall of famer who was an elite defender but a below average hitter is Ozzie Smith.
Unfortunately, the fielding statistics provided by the Lahman data set leave much to be desired.
The defensive stats listed here are not substantial enough to include.
For these reasons, defensive metrics will not be included in this model.

In [9]:
import numpy as np

fielding_df = lahman_dfs["Fielding"]
fielding_df


,playerID,yearID,stint,teamID,lgID,POS,G,GS,InnOuts,PO,A,E,DP,PB,WP,SB,CS,ZR
0,abercda01,1871,1,TRO,NaN,SS,1,1.0,24.0,1,3,2.0,0,NaN,NaN,NaN,NaN,NaN
1,addybo01,1871,1,RC1,NaN,2B,22,22.0,606.0,67,72,42.0,5,NaN,NaN,NaN,NaN,NaN
2,addybo01,1871,1,RC1,NaN,SS,3,3.0,96.0,8,14,7.0,0,NaN,NaN,NaN,NaN,NaN
3,allisar01,1871,1,CL1,NaN,2B,2,0.0,18.0,1,4,0.0,0,NaN,NaN,NaN,NaN,NaN
4,allisar01,1871,1,CL1,NaN,OF,29,29.0,729.0,51,3,7.0,1,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147075,zimmejo02,2021,1,MIL,NL,P,2,0.0,17.0,0,1,0.0,0,NaN,NaN,0.0,0.0,NaN
147076,zimmeky01,2021,1,KCA,AL,P,52,2.0,162.0,4,5,0.0,3,NaN,NaN,2.0,1.0,NaN
147077,zimmery01,2021,1,WAS,NL,1B,54,45.0,1202.0,338,15,0.0,30,NaN,NaN,NaN,NaN,NaN
147078,zuberty01,2021,1,KCA,AL,P,31,0.0,82.0,1,3,0.0,0,NaN,NaN,0.0,0.0,NaN


# Awards

Next, we'll tally up awards that players got.
There are two CSVs of interest, AwardsPlayers.csv and AwardsSharePlayers.csv.
Both of these are not fully updated to the most current season, which at the time of writing is 2021.

In [10]:
import numpy as np

awards_df = lahman_dfs["AwardsPlayers"]
awards_df["award"] = np.where(awards_df["awardID"].isna(), 0, 1)
awards_df = awards_df.groupby(by=["playerID"]).sum()
awards_df = awards_df.drop(labels=["yearID"], axis=1)
awards_df


,award
playerID,
aaronha01,16
abbotji01,2
abernte02,2
abreubo01,2
abreujo02,7
...,...
zimmehe01,5
zimmery01,5
ziskri01,2


In [11]:
awards_sh_df = lahman_dfs["AwardsSharePlayers"]
awards_sh_df["award"] = np.where(awards_sh_df["awardID"].isna(), 0, 1)
awards_sh_df = awards_sh_df.groupby(by=["playerID"]).sum()
awards_sh_df = awards_sh_df.drop(
    labels=["yearID", "pointsWon", "pointsMax", "votesFirst"], axis=1
)
awards_sh_df


,award
playerID,
aaronha01,20
abbotji01,2
abernte02,1
abramca01,1
abreubo01,7
...,...
zimmejo02,2
zimmery01,4
ziskri01,4


Now we will merge these two awards dataframes.

In [12]:
awards_full_df = pd.merge(
    awards_df,
    awards_sh_df,
    on="playerID",
    how="outer",
    suffixes=["_left", "_right"],
)

# Fill null values with 0
awards_full_df = awards_full_df.fillna(0)

# Sum up all awards
awards_full_df["awardsCount"] = (
    awards_full_df["award_left"] + awards_full_df["award_right"]
)

# Drop left and right labels and reset index
awards_full_df = awards_full_df.drop(labels=["award_left", "award_right"], axis=1)
awards_full_df = awards_full_df.reset_index()

awards_full_df


,playerID,awardsCount
0,aaronha01,36.0
1,abbotji01,4.0
2,abernte02,3.0
3,abreubo01,9.0
4,abreujo02,10.0
...,...,...
2680,zeileto01,1.0
2681,zernigu01,2.0
2682,zieglbr01,1.0
2683,zimmeje02,1.0


# People

We will get the names of the players based on their `playerID`.
This will not help creating the training data, instead, it will simply help knowing who the players are.

In [13]:
people_df = lahman_dfs["People"][["playerID", "nameLast", "nameFirst"]]


In [14]:
people_df.describe()


,playerID,nameLast,nameFirst
count,20370,20370,20333
unique,20370,10331,2573
top,aardsda01,Smith,Bill
freq,1,167,548


# Training Data

Now we are mostly there to create our training data.
We needd to join the tables, `hof_df`, `batting_df` and `awards_full_df`, together.
This will be used for our training data.

In [15]:
# Left join between hof_df and batters_df, want to keep all the features in the hof_df
joined_df = (
    hof_df.set_index("playerID").join(batters_df.set_index("playerID")).reset_index()
)
assert joined_df.shape[0] == hof_df.shape[0]

# Merge in the awards count
joined_df = (
    joined_df.set_index("playerID")
    .join(awards_full_df.set_index("playerID"))
    .reset_index()
)

# If the awards count is null, then replace with 0
joined_df["awardsCount"] = np.where(
    joined_df["awardsCount"].isna(), 0, joined_df["awardsCount"]
)

# Add in names
joined_df = (
    joined_df.set_index("playerID").join(people_df.set_index("playerID")).reset_index()
)

joined_df


,playerID,yearID,votedBy,ballots,needed,votes,inducted,category,G,AB,...,IBB,HBP,SF,OBP,SLG,OPS,BA,awardsCount,nameLast,nameFirst
0,cobbty01,1936,BBWAA,226.0,170.0,222.0,Y,Player,3035.0,11436.0,...,0.0,94.0,0.0,0.432898,0.511892,0.944790,0.366299,35.0,Cobb,Ty
1,doyleja01,1936,Veterans,78.0,59.0,1.0,N,Player,1569.0,6055.0,...,0.0,49.0,0.0,0.351467,0.384806,0.736273,0.299092,0.0,Doyle,Jack
2,bondto01,1936,Veterans,78.0,59.0,1.0,N,Player,488.0,1975.0,...,0.0,0.0,0.0,0.246870,0.276456,0.523326,0.238481,1.0,Bond,Tommy
3,battijo01,1936,Veterans,78.0,59.0,1.0,N,Player,480.0,1953.0,...,0.0,1.0,0.0,0.240722,0.281106,0.521828,0.224782,0.0,Battin,Joe
4,allisdo01,1936,Veterans,78.0,59.0,1.0,N,Player,318.0,1407.0,...,0.0,0.0,0.0,0.283718,0.321251,0.604969,0.271500,0.0,Allison,Doug
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1207,ramirma02,2018,BBWAA,422.0,317.0,93.0,N,Player,2302.0,8244.0,...,216.0,109.0,90.0,0.410561,0.585395,0.995956,0.312227,31.0,Ramirez,Manny
1208,kentje01,2018,BBWAA,422.0,317.0,61.0,N,Player,2298.0,8498.0,...,61.0,125.0,103.0,0.355516,0.499647,0.855163,0.289598,15.0,Kent,Jeff
1209,sheffga01,2018,BBWAA,422.0,317.0,47.0,N,Player,2576.0,9217.0,...,130.0,135.0,111.0,0.393033,0.513942,0.906975,0.291744,16.0,Sheffield,Gary
1210,rolensc01,2018,BBWAA,422.0,317.0,43.0,N,Player,2038.0,7398.0,...,57.0,127.0,93.0,0.364330,0.490403,0.854733,0.280752,18.0,Rolen,Scott


In [16]:
import numpy as np

# New label indicating whether player was inducted into HOF, binary value
joined_df["HOF"] = np.where(joined_df["inducted"] == "Y", 1, 0)

joined_df = joined_df.sort_values(by=["playerID"]).reset_index(drop=True)

joined_df


,playerID,yearID,votedBy,ballots,needed,votes,inducted,category,G,AB,...,HBP,SF,OBP,SLG,OPS,BA,awardsCount,nameLast,nameFirst,HOF
0,aaronha01,1982,BBWAA,415.0,312.0,406.0,Y,Player,3298.0,12364.0,...,32.0,121.0,0.373949,0.554513,0.928462,0.304998,36.0,Aaron,Hank,1
1,abbotji01,2005,BBWAA,516.0,387.0,13.0,N,Player,263.0,21.0,...,0.0,0.0,0.095238,0.095238,0.190476,0.095238,4.0,Abbott,Jim,0
2,adamsba01,1955,BBWAA,251.0,189.0,24.0,N,Player,482.0,1019.0,...,1.0,0.0,0.251631,0.280667,0.532298,0.211973,4.0,Adams,Babe,0
3,adamsbo03,1966,BBWAA,302.0,227.0,1.0,N,Player,1281.0,4019.0,...,17.0,5.0,0.339618,0.368002,0.707620,0.269221,1.0,Adams,Bobby,0
4,adamssp01,1960,BBWAA,269.0,202.0,1.0,N,Player,1424.0,5557.0,...,28.0,0.0,0.342663,0.352708,0.695371,0.285766,1.0,Adams,Sparky,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1207,zahnge01,1991,BBWAA,443.0,333.0,0.0,N,Player,304.0,43.0,...,0.0,0.0,0.139535,0.139535,0.279070,0.139535,2.0,Zahn,Geoff,0
1208,zambrca01,2018,BBWAA,422.0,317.0,0.0,N,Player,384.0,693.0,...,0.0,4.0,0.247525,0.388167,0.635692,0.238095,7.0,Zambrano,Carlos,0
1209,zeileto01,2010,BBWAA,539.0,405.0,0.0,N,Player,2158.0,7573.0,...,42.0,81.0,0.346140,0.423346,0.769487,0.264624,1.0,Zeile,Todd,0
1210,zimmech01,1938,BBWAA,262.0,197.0,1.0,N,Player,1280.0,4546.0,...,91.0,0.0,0.339367,0.368896,0.708263,0.269468,0.0,Zimmer,Chief,0


Drop rows that we don't need.
Also look for any rows that have null values.

In [17]:
training_df = joined_df.drop(
    labels=[
        "votedBy",
        "ballots",
        "needed",
        "votes",
        "inducted",
        "category",
    ],
    axis=1,
)

# Describe the data frame, want to see where we have null rows
training_df.describe()


,yearID,G,AB,R,H,2B,3B,HR,RBI,SB,...,SO,IBB,HBP,SF,OBP,SLG,OPS,BA,awardsCount,HOF
count,1212.000000,1186.000000,1186.000000,1186.000000,1186.000000,1186.000000,1186.000000,1186.000000,1186.000000,1186.000000,...,1186.000000,1186.000000,1186.000000,1186.000000,1178.000000,1178.000000,1178.000000,1178.000000,1212.000000,1212.000000
mean,1981.323432,1285.511804,4139.370152,597.532884,1147.845700,197.629005,40.177909,103.507589,557.946037,92.900506,...,525.582631,28.158516,28.749578,22.704890,0.298331,0.348117,0.646448,0.239493,7.195545,0.196370
std,23.648982,725.289083,3098.413608,514.407910,928.423731,168.979061,45.194689,133.451021,494.304090,143.652130,...,451.279240,50.279474,35.234484,31.460949,0.083065,0.122654,0.201834,0.067198,8.811760,0.397415
min,1936.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1960.000000,578.250000,888.250000,71.000000,170.250000,23.000000,4.000000,4.000000,70.250000,2.000000,...,182.000000,0.000000,2.250000,0.000000,0.235742,0.252478,0.485111,0.195324,1.000000,0.000000
50%,1983.000000,1313.500000,4503.500000,594.500000,1216.000000,195.500000,27.000000,42.000000,511.000000,36.000000,...,371.500000,0.000000,19.000000,1.000000,0.327249,0.375117,0.709440,0.262955,4.000000,0.000000
75%,2001.000000,1853.000000,6653.000000,973.000000,1875.750000,321.000000,59.000000,164.000000,923.000000,125.000000,...,780.500000,43.750000,40.000000,44.000000,0.357999,0.441000,0.797174,0.285294,10.000000,0.000000
max,2018.000000,3562.000000,14053.000000,2295.000000,4256.000000,792.000000,309.000000,762.000000,2297.000000,1406.000000,...,2597.000000,688.000000,287.000000,128.000000,0.500000,0.689807,1.163767,0.500000,63.000000,1.000000


We can see from above that there are a few rows that have null values.
Let's take a look at these values and see if there is any kind of pattern.

In [18]:
training_df[training_df["BA"].isna() | training_df["OPS"].isna()]


,playerID,yearID,G,AB,R,H,2B,3B,HR,RBI,...,HBP,SF,OBP,SLG,OPS,BA,awardsCount,nameLast,nameFirst,HOF
62,bellco99,1974,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Bell,Cool Papa,1
93,boddimi01,1999,346.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,6.0,Boddicker,Mike,0
128,brownra99,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Brown,Ray,1
193,charlos99,1976,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Charleston,Oscar,1
223,coopean99,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Cooper,Andy,1
251,dandrra99,1987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Dandridge,Ray,1
269,dayle99,1995,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Day,Leon,1
282,dihigma99,1977,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Dihigo,Martin,1
350,flanami01,1998,529.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,5.0,Flanagan,Mike,0
360,fostebi99,1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Foster,Bill,1


I took the liberty to look up some of these players on [SportsReference](https://www.baseball-reference.com/).
Some of the players in this list are pitchers.
A huge chunk of these players are from the Negro Leagues.
These players would be unsuitable to use in training data since they would be seen as outliers.
Many of them didn't play that many games and we want to make sure we training on data that is most representative of the average HOF batter.
For these reasons, we will simply remove all of the data points in the above DataFrame from our training data.
There are a few other outliers, such as any other pitchers that have slipped through the cracks, and players that won't get voted in due to steroids usage (such as Barry Bonds, Manny Ramirez, Jason Giambi, Sammy Sosa).

In [19]:
# Drop rows we do not have stats for
training_df = training_df.drop(
    training_df[training_df["BA"].isna() | training_df["OPS"].isna()].index
).reset_index(drop=True)
training_df


,playerID,yearID,G,AB,R,H,2B,3B,HR,RBI,...,HBP,SF,OBP,SLG,OPS,BA,awardsCount,nameLast,nameFirst,HOF
0,aaronha01,1982,3298.0,12364.0,2174.0,3771.0,624.0,98.0,755.0,2297.0,...,32.0,121.0,0.373949,0.554513,0.928462,0.304998,36.0,Aaron,Hank,1
1,abbotji01,2005,263.0,21.0,0.0,2.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.095238,0.095238,0.190476,0.095238,4.0,Abbott,Jim,0
2,adamsba01,1955,482.0,1019.0,79.0,216.0,31.0,15.0,3.0,75.0,...,1.0,0.0,0.251631,0.280667,0.532298,0.211973,4.0,Adams,Babe,0
3,adamsbo03,1966,1281.0,4019.0,591.0,1082.0,188.0,49.0,37.0,303.0,...,17.0,5.0,0.339618,0.368002,0.707620,0.269221,1.0,Adams,Bobby,0
4,adamssp01,1960,1424.0,5557.0,844.0,1588.0,249.0,48.0,9.0,394.0,...,28.0,0.0,0.342663,0.352708,0.695371,0.285766,1.0,Adams,Sparky,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1173,zahnge01,1991,304.0,43.0,3.0,6.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.139535,0.139535,0.279070,0.139535,2.0,Zahn,Geoff,0
1174,zambrca01,2018,384.0,693.0,75.0,165.0,26.0,3.0,24.0,71.0,...,0.0,4.0,0.247525,0.388167,0.635692,0.238095,7.0,Zambrano,Carlos,0
1175,zeileto01,2010,2158.0,7573.0,986.0,2004.0,397.0,23.0,253.0,1110.0,...,42.0,81.0,0.346140,0.423346,0.769487,0.264624,1.0,Zeile,Todd,0
1176,zimmech01,1938,1280.0,4546.0,617.0,1225.0,222.0,76.0,26.0,625.0,...,91.0,0.0,0.339367,0.368896,0.708263,0.269468,0.0,Zimmer,Chief,0


In [20]:
training_in_hof_df = training_df.loc[training_df["HOF"] == 1]
training_nin_hof_df = training_df.loc[training_df["HOF"] == 0]

print(f"There are {training_in_hof_df.shape[0]} players in the hof in our training set")
print(
    f"There are {training_nin_hof_df.shape[0]} players not in the hof in our training set"
)


There are 212 players in the hof in our training set
There are 966 players not in the hof in our training set


In [21]:
training_in_hof_df.sort_values(by=["G"], ascending=True).head(25)


,playerID,yearID,G,AB,R,H,2B,3B,HR,RBI,...,HBP,SF,OBP,SLG,OPS,BA,awardsCount,nameLast,nameFirst,HOF
127,brownwi02,2006,21.0,67.0,4.0,12.0,3.0,0.0,1.0,6.0,...,0.0,0.0,0.179104,0.268657,0.447761,0.179104,0.0,Brown,Willard,1
800,paigesa01,1971,179.0,124.0,2.0,12.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.118110,0.096774,0.214884,0.096774,1.0,Paige,Satchel,1
543,jossad01,1978,296.0,817.0,46.0,118.0,21.0,6.0,1.0,51.0,...,1.0,0.0,0.198394,0.188494,0.386889,0.144431,1.0,Joss,Addie,1
263,deandi01,1953,325.0,717.0,76.0,161.0,23.0,4.0,8.0,76.0,...,1.0,0.0,0.235213,0.301255,0.536468,0.224547,19.0,Dean,Dizzy,1
390,gomezle01,1972,368.0,904.0,59.0,133.0,11.0,0.0,0.0,58.0,...,2.0,0.0,0.193515,0.159292,0.352807,0.147124,15.0,Gomez,Lefty,1
191,chesbja01,1946,393.0,1103.0,85.0,217.0,37.0,13.0,5.0,82.0,...,2.0,0.0,0.215929,0.267452,0.483382,0.196736,0.0,Chesbro,Jack,1
586,koufasa01,1972,397.0,776.0,26.0,75.0,9.0,0.0,2.0,28.0,...,1.0,3.0,0.144593,0.115979,0.260572,0.096649,32.0,Koufax,Sandy,1
1076,vanceda01,1955,442.0,971.0,68.0,146.0,23.0,1.0,7.0,75.0,...,7.0,0.0,0.219489,0.197734,0.417223,0.150360,14.0,Vance,Dazzy,1
224,covelst01,1969,451.0,1058.0,70.0,168.0,26.0,10.0,1.0,81.0,...,4.0,0.0,0.201794,0.205104,0.406898,0.158790,4.0,Coveleski,Stan,1
656,maricju01,1983,475.0,1221.0,73.0,202.0,29.0,2.0,4.0,75.0,...,2.0,7.0,0.190852,0.202293,0.393145,0.165438,13.0,Marichal,Juan,1


We will need to cut down both sets further.
Since we are only looking at hitters, we need to make sure that the data resembles that.
The easiest way to prune out pitchers would be add a further stipulation that they had to have `x` number of at-bats.
We will use 1000 for now.

In [22]:
# Need to have been in the league for at least 10 years.
# We are only concerned with batters, so they must have a significant amount of games where they hit
training_in_hof_df = (
    training_in_hof_df.loc[training_in_hof_df["G"] > 1100]
    .sort_values(by=["G"], ascending=True)
    .reset_index(drop=True)
)

print(f"There are {training_in_hof_df.shape[0]} players in the hof in our training set")


There are 146 players in the hof in our training set


In [23]:
training_in_hof_df.sort_values(by=["G"], ascending=True).head(25)


,playerID,yearID,G,AB,R,H,2B,3B,HR,RBI,...,HBP,SF,OBP,SLG,OPS,BA,awardsCount,nameLast,nameFirst,HOF
0,youngro01,1972,1211.0,4627.0,812.0,1491.0,236.0,93.0,42.0,592.0,...,37.0,0.0,0.398542,0.440674,0.839217,0.322239,6.0,Youngs,Ross,1
1,camparo01,1969,1215.0,4205.0,627.0,1161.0,178.0,18.0,242.0,856.0,...,30.0,18.0,0.360217,0.499643,0.859861,0.276100,17.0,Campanella,Roy,1
2,mccarto01,1946,1273.0,5120.0,1066.0,1493.0,191.0,53.0,44.0,732.0,...,50.0,0.0,0.364353,0.375391,0.739744,0.291602,0.0,McCarthy,Tommy,1
3,hafeych01,1971,1283.0,4625.0,777.0,1466.0,341.0,67.0,164.0,833.0,...,33.0,0.0,0.371968,0.526054,0.898022,0.316973,8.0,Hafey,Chick,1
4,ewingbu01,1939,1315.0,5363.0,1129.0,1625.0,250.0,178.0,71.0,883.0,...,9.0,0.0,0.351492,0.455715,0.807207,0.303002,0.0,Ewing,Buck,1
5,wilsoha01,1979,1348.0,4760.0,884.0,1461.0,266.0,67.0,244.0,1063.0,...,20.0,0.0,0.395123,0.544748,0.939871,0.306933,13.0,Wilson,Hack,1
6,robinja02,1962,1382.0,4877.0,947.0,1518.0,273.0,54.0,137.0,734.0,...,72.0,9.0,0.408915,0.473652,0.882567,0.311257,19.0,Robinson,Jackie,1
7,greenha01,1956,1394.0,5193.0,1051.0,1628.0,379.0,71.0,331.0,1276.0,...,16.0,0.0,0.411813,0.605045,1.016858,0.313499,20.0,Greenberg,Hank,1
8,thompsa01,1974,1410.0,5998.0,1261.0,1988.0,343.0,161.0,126.0,1308.0,...,63.0,0.0,0.384308,0.505335,0.889643,0.331444,0.0,Thompson,Sam,1
9,lindsfr01,1976,1438.0,5611.0,895.0,1747.0,301.0,81.0,103.0,779.0,...,13.0,0.0,0.351460,0.448940,0.800400,0.311353,6.0,Lindstrom,Freddie,1


The reason why we won't do something similar for the not in hof group is because individuals with low game counts should are still valid training points. Instead, we will take out outliers who statistically should be in the HOF, but aren't due to steroid usage.

In [24]:
training_nin_hof_df.sort_values(by=["HR"], ascending=False).head(25)

,playerID,yearID,G,AB,R,H,2B,3B,HR,RBI,...,HBP,SF,OBP,SLG,OPS,BA,awardsCount,nameLast,nameFirst,HOF
96,bondsba01,2018,2986.0,9847.0,2227.0,2935.0,601.0,77.0,762.0,1996.0,...,106.0,91.0,0.444295,0.606885,1.051180,0.298060,63.0,Bonds,Barry,0
990,sosasa01,2018,2354.0,8813.0,1475.0,2408.0,379.0,45.0,609.0,1667.0,...,59.0,78.0,0.343759,0.533757,0.877516,0.273233,25.0,Sosa,Sammy,0
698,mcgwima01,2016,1874.0,6187.0,1167.0,1626.0,252.0,6.0,583.0,1414.0,...,75.0,78.0,0.394149,0.588169,0.982318,0.262809,20.0,McGwire,Mark,0
802,palmera01,2014,2831.0,10472.0,1663.0,3020.0,585.0,38.0,569.0,1835.0,...,87.0,119.0,0.370709,0.514515,0.885224,0.288388,18.0,Palmeiro,Rafael,0
851,ramirma02,2018,2302.0,8244.0,1544.0,2574.0,547.0,20.0,555.0,1831.0,...,109.0,90.0,0.410561,0.585395,0.995956,0.312227,31.0,Ramirez,Manny,0
964,sheffga01,2018,2576.0,9217.0,1636.0,2689.0,467.0,27.0,509.0,1676.0,...,135.0,111.0,0.393033,0.513942,0.906975,0.291744,16.0,Sheffield,Gary,0
697,mcgrifr01,2018,2460.0,8757.0,1349.0,2490.0,441.0,24.0,493.0,1550.0,...,39.0,71.0,0.376917,0.509078,0.885995,0.284344,15.0,McGriff,Fred,0
266,delgaca01,2015,2035.0,7283.0,1241.0,2038.0,483.0,18.0,473.0,1512.0,...,172.0,93.0,0.383389,0.545929,0.929318,0.279830,15.0,Delgado,Carlos,0
160,cansejo01,2007,1887.0,7057.0,1186.0,1877.0,340.0,14.0,462.0,1407.0,...,84.0,81.0,0.352731,0.514525,0.867256,0.265977,18.0,Canseco,Jose,0
573,kingmda01,1992,1941.0,6677.0,901.0,1575.0,240.0,25.0,442.0,1210.0,...,53.0,75.0,0.301632,0.477909,0.779542,0.235884,7.0,Kingman,Dave,0


In [25]:
training_nin_hof_df = (
    training_nin_hof_df.set_index("playerID")
    .drop(
        index=[
            "bondsba01",
            "sosasa01",
            "mcgwima01",
            "palmera01",
            "ramirma02",
            "sheffga01",
            "cansejo01",
        ]
    )
    .reset_index()
)
training_nin_hof_df


,playerID,yearID,G,AB,R,H,2B,3B,HR,RBI,...,HBP,SF,OBP,SLG,OPS,BA,awardsCount,nameLast,nameFirst,HOF
0,abbotji01,2005,263.0,21.0,0.0,2.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.095238,0.095238,0.190476,0.095238,4.0,Abbott,Jim,0
1,adamsba01,1955,482.0,1019.0,79.0,216.0,31.0,15.0,3.0,75.0,...,1.0,0.0,0.251631,0.280667,0.532298,0.211973,4.0,Adams,Babe,0
2,adamsbo03,1966,1281.0,4019.0,591.0,1082.0,188.0,49.0,37.0,303.0,...,17.0,5.0,0.339618,0.368002,0.707620,0.269221,1.0,Adams,Bobby,0
3,adamssp01,1960,1424.0,5557.0,844.0,1588.0,249.0,48.0,9.0,394.0,...,28.0,0.0,0.342663,0.352708,0.695371,0.285766,1.0,Adams,Sparky,0
4,ageeto01,1979,1129.0,3912.0,558.0,999.0,170.0,27.0,130.0,433.0,...,34.0,15.0,0.319545,0.412321,0.731866,0.255368,7.0,Agee,Tommie,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
954,zahnge01,1991,304.0,43.0,3.0,6.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.139535,0.139535,0.279070,0.139535,2.0,Zahn,Geoff,0
955,zambrca01,2018,384.0,693.0,75.0,165.0,26.0,3.0,24.0,71.0,...,0.0,4.0,0.247525,0.388167,0.635692,0.238095,7.0,Zambrano,Carlos,0
956,zeileto01,2010,2158.0,7573.0,986.0,2004.0,397.0,23.0,253.0,1110.0,...,42.0,81.0,0.346140,0.423346,0.769487,0.264624,1.0,Zeile,Todd,0
957,zimmech01,1938,1280.0,4546.0,617.0,1225.0,222.0,76.0,26.0,625.0,...,91.0,0.0,0.339367,0.368896,0.708263,0.269468,0.0,Zimmer,Chief,0


In [26]:
training = pd.concat(
    [
        training_in_hof_df.set_index("playerID"),
        training_nin_hof_df.set_index("playerID"),
    ]
)
training


,yearID,G,AB,R,H,2B,3B,HR,RBI,SB,...,HBP,SF,OBP,SLG,OPS,BA,awardsCount,nameLast,nameFirst,HOF
playerID,,,,,,,,,,,,,,,,,,,,,
youngro01,1972,1211.0,4627.0,812.0,1491.0,236.0,93.0,42.0,592.0,153.0,...,37.0,0.0,0.398542,0.440674,0.839217,0.322239,6.0,Youngs,Ross,1
camparo01,1969,1215.0,4205.0,627.0,1161.0,178.0,18.0,242.0,856.0,25.0,...,30.0,18.0,0.360217,0.499643,0.859861,0.276100,17.0,Campanella,Roy,1
mccarto01,1946,1273.0,5120.0,1066.0,1493.0,191.0,53.0,44.0,732.0,468.0,...,50.0,0.0,0.364353,0.375391,0.739744,0.291602,0.0,McCarthy,Tommy,1
hafeych01,1971,1283.0,4625.0,777.0,1466.0,341.0,67.0,164.0,833.0,70.0,...,33.0,0.0,0.371968,0.526054,0.898022,0.316973,8.0,Hafey,Chick,1
ewingbu01,1939,1315.0,5363.0,1129.0,1625.0,250.0,178.0,71.0,883.0,354.0,...,9.0,0.0,0.351492,0.455715,0.807207,0.303002,0.0,Ewing,Buck,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zahnge01,1991,304.0,43.0,3.0,6.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.139535,0.139535,0.279070,0.139535,2.0,Zahn,Geoff,0
zambrca01,2018,384.0,693.0,75.0,165.0,26.0,3.0,24.0,71.0,1.0,...,0.0,4.0,0.247525,0.388167,0.635692,0.238095,7.0,Zambrano,Carlos,0
zeileto01,2010,2158.0,7573.0,986.0,2004.0,397.0,23.0,253.0,1110.0,53.0,...,42.0,81.0,0.346140,0.423346,0.769487,0.264624,1.0,Zeile,Todd,0


# Training the Data

We need to make sure that when training the data, there are an equal number of examples of players that are in the hof and those that are not.
If we have an overwhelming majority in one category, then the model can simply just pick that category every time.
For example, 99% of players of do not make it to the Hall of Fame.
A very naive, yet accurate model, would be to say that a specified player would not make it into the HoF, and the model would be right 99% of the time.

In [27]:
print(
    f'There are {training.loc[training["HOF"] == 1].shape[0]} players in the hof in our training set'
)
print(
    f'There are {training.loc[training["HOF"] == 0].shape[0]} players not in the hof in our training set'
)


There are 146 players in the hof in our training set
There are 959 players not in the hof in our training set


In order to overcome this imbalance, we will randomly undersample from the players that are not in the hof.

In [28]:
training.columns


Index(['yearID', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS',
       'BB', 'SO', 'IBB', 'HBP', 'SF', 'OBP', 'SLG', 'OPS', 'BA',
       'awardsCount', 'nameLast', 'nameFirst', 'HOF'],
      dtype='object')

In [29]:
training = training.drop(labels=["yearID", "nameLast", "nameFirst"], axis=1)

In [30]:
X = pd.concat(
    [
        training.loc[training["HOF"] == 1],
        training.loc[training["HOF"] == 0].sample(146, random_state=27),
    ]
)

print(
    f'There are {X.loc[X["HOF"] == 1].shape[0]} players in the hof in our training set'
)
print(
    f'There are {X.loc[X["HOF"] == 0].shape[0]} players not in the hof in our training set'
)


There are 146 players in the hof in our training set
There are 146 players not in the hof in our training set


In [31]:
import pandas as pd

from sklearn.model_selection import train_test_split

y = X.filter(items=["HOF"], axis=1).reset_index(drop=True)
X = X.drop(
    labels=["HOF", "2B", "3B", "SO", "IBB", "HBP", "SF", "CS"], axis=1
).reset_index(drop=True)


In [32]:
X


,G,AB,R,H,HR,RBI,SB,BB,OBP,SLG,OPS,BA,awardsCount
0,1211.0,4627.0,812.0,1491.0,42.0,592.0,153.0,550.0,0.398542,0.440674,0.839217,0.322239,6.0
1,1215.0,4205.0,627.0,1161.0,242.0,856.0,25.0,533.0,0.360217,0.499643,0.859861,0.276100,17.0
2,1273.0,5120.0,1066.0,1493.0,44.0,732.0,468.0,536.0,0.364353,0.375391,0.739744,0.291602,0.0
3,1283.0,4625.0,777.0,1466.0,164.0,833.0,70.0,372.0,0.371968,0.526054,0.898022,0.316973,8.0
4,1315.0,5363.0,1129.0,1625.0,71.0,883.0,354.0,392.0,0.351492,0.455715,0.807207,0.303002,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,324.0,413.0,20.0,45.0,0.0,13.0,0.0,14.0,0.148148,0.135593,0.283741,0.108959,0.0
288,1553.0,4908.0,541.0,1227.0,146.0,599.0,11.0,407.0,0.307220,0.386716,0.693936,0.250000,2.0
289,1465.0,5520.0,761.0,1593.0,126.0,663.0,196.0,472.0,0.344265,0.421196,0.765461,0.288587,4.0
290,2030.0,6754.0,990.0,1977.0,95.0,920.0,65.0,820.0,0.372307,0.415606,0.787912,0.292715,12.0


In [33]:
y

,HOF
0,1
1,1
2,1
3,1
4,1
...,...
287,0
288,0
289,0
290,0


In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=27, shuffle=True
)


## Logistic Regression

In [35]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(solver="liblinear", random_state=27, multi_class="ovr").fit(
    X_train, y_train.values.ravel()
)


In [36]:
clf.score(X_train, y_train)


0.927038626609442

In [37]:
clf.score(X_test, y_test)


0.8813559322033898

In [38]:
clf.score(X, y)


0.9178082191780822

In [39]:
pd.DataFrame(clf.coef_, columns=clf.feature_names_in_)

,G,AB,R,H,HR,RBI,SB,BB,OBP,SLG,OPS,BA,awardsCount
0,-0.008796,0.000017,-0.000705,0.005338,-0.013211,0.008858,0.000781,0.003579,-0.568831,-0.676698,-1.245529,-0.458262,0.171554


In [40]:
clf.predict_proba(
    training_df.loc[training_df["playerID"] == "trammal01"].drop(
        labels=[
            "HOF",
            "nameFirst",
            "nameLast",
            "playerID",
            "yearID",
            "2B",
            "3B",
            "SO",
            "IBB",
            "HBP",
            "SF",
            "CS",
        ],
        axis=1,
    )
)


array([[0.22207901, 0.77792099]])

## Random Forests

In [41]:
from sklearn.ensemble import RandomForestClassifier

RF = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=27).fit(
    X_train, y_train.values.ravel()
)
RF.score(X_train, y_train)


0.9356223175965666

In [42]:
RF.score(X_test, y_test)


0.9152542372881356

In [43]:
RF.score(X, y)


0.9315068493150684

In [44]:
pd.DataFrame(
    RF.feature_importances_.reshape(1, RF.feature_importances_.shape[0]),
    columns=RF.feature_names_in_,
)


,G,AB,R,H,HR,RBI,SB,BB,OBP,SLG,OPS,BA,awardsCount
0,0.019346,0.066327,0.15622,0.161573,0.010745,0.148414,0.004711,0.013609,0.070849,0.037494,0.084675,0.107244,0.118792


In [45]:
RF.predict_proba(
    training_df.loc[training_df["playerID"] == "trammal01"].drop(
        labels=[
            "HOF",
            "nameFirst",
            "nameLast",
            "playerID",
            "yearID",
            "2B",
            "3B",
            "SO",
            "IBB",
            "HBP",
            "SF",
            "CS",
        ],
        axis=1,
    )
)


array([[0.1260145, 0.8739855]])